In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
import sys, os

# 本地自写回测引擎 + 本地 DuckDB 数据源（data_source 默认后端离线读 data/ 下 parquet，与平台对拍一致）
# 定位本工程根（含 engine/）：data_source / smalltrader 在 engine/ 下，从 cwd 向上找根目录并入 sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, 'engine')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
sys.path.insert(0, os.path.join(_root, 'engine'))
sys.path.insert(0, _root)
import smalltrader as bigtrader
import data_source   # 注册自动取数：bigtrader.run(data=缺省) 时按 market/instruments 从本地 DuckDB 拉行情（对齐平台原版隐式取数）

import queries       # 本地 DuckDB：cn_stock_prefactors 等视图（表清单见 queries.TABLES，惰性读 data/ 下 parquet）
con = queries.connect()

today = datetime.now().strftime("%Y-%m-%d")
START_DATE = '2021-01-01'
END_DATE   = '2026-09-03'
CAPITAL    = 1000000

def initialize(context: bigtrader.IContext):
    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(bigtrader.PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    #  context.data = signal_df
    context.holding_days = 3
    context.target_hold_count = 20

    context.sql = """
    WITH ranked AS (
        SELECT
            date, instrument,
            percent_rank() OVER (PARTITION BY date ORDER BY total_market_cap)   AS score1,
            percent_rank() OVER (PARTITION BY date ORDER BY net_profit_mrq_yoy) AS score3
        FROM cn_stock_prefactors
        WHERE 
        is_risk_warning = 0
        AND list_days >= 252
        AND st_status = 0
        AND list_sector = 1
        AND date BETWEEN ? AND ?
        AND total_market_cap IS NOT NULL
        AND net_profit_mrq_yoy IS NOT NULL  
        AND date IS NOT NULL 
    )
    SELECT date, instrument, score1, score3, score1 - score3 AS score FROM ranked
    """

    context.data = con.execute(context.sql,
        [context.add_trading_days(context.start_date, -250), context.end_date]).df().dropna()


def befor_trading(context, data):
    pass


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    # 每 context.holding_days 个交易日调仓一次
    if context.trading_day_index % context.holding_days != 0:
        return

    # 获取当前日期
    current_date = data.current_dt.strftime("%Y-%m-%d")
    # 获取当日数据
    current_day_data = context.data[context.data["date"] == current_date]
    # 取前10只
    current_day_data = current_day_data.assign(score=current_day_data['score'].round(10))
    current_day_data.sort_values(by=['score', 'instrument'], inplace=True, ascending=True)
    current_day_data = current_day_data.head(context.target_hold_count)
    len_ = len(current_day_data)
    # 获取当日目标持有股票
    target_hold_instruments = set(current_day_data["instrument"])

    # 获取当前已持有股票
    current_hold_instruments = set(context.get_account_positions().keys())

    # 卖出不在目标持有列表中的股票
    for instrument in current_hold_instruments - target_hold_instruments:
        context.order_target_percent(instrument, 0)

    # 买入目标持有列表中的股票
    for instrument in target_hold_instruments - current_hold_instruments:
        context.order_target_percent(instrument, 0.9 / len_)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=END_DATE,
    capital_base=CAPITAL,            # 设置初始资金
    initialize=initialize,           # 传入初始化函数
    handle_data=handle_data,         # 传入数据处理函数
    before_trading_start=befor_trading,
    #instruments=universe,            # 可交易池（本地 DuckDB 自动取数：行情/除权除息/名称/基准）
    order_price_field_buy='open',
    order_price_field_sell='open',
    render="report"        # 渲染绩效报告，展示回测结果 可选None/"auto"/"report"
)

# 渲染绩效报告，展示回测结果
#datadf = performance.read_raw_perf()
#performance.render()

## BigTrader 回测报告

### 绩效摘要

| 指标    | 值 |
|-------|------|
| 累计收益率 | 359.67% |
| 年化收益率 | 32.25% |
| 基准收益率 | -13.58% |
| 阿尔法   | 0.35 |
| 贝塔    | 0.53 |
| 夏普比率  | 1.18 |
| 胜率    | 62.75% |
| 盈亏比   | 1.41 |
| 收益波动率 | 23.65% |
| 信息比率  | 1.37 |
| 最大回撤  | 31.64% |

### 收益走势

| 日期 | 策略收益率 | 基准收益率 | 相对收益率 |
|------|----------|----------|----------|
| 2021-01-04 | 0.0% | 0.0% | 0.0% |
| 2021-01-29 | -7.37% | 1.6% | -9.22% |
| 2021-02-26 | -3.71% | 1.31% | -5.82% |
| 2021-03-31 | 2.77% | -4.16% | 5.68% |
| 2021-04-30 | -2.43% | -2.74% | -1.31% |
| 2021-05-31 | 5.9% | 1.21% | 2.73% |
| 2021-06-30 | 9.77% | -0.83% | 8.59% |
| 2021-07-30 | 9.3% | -8.67% | 17.01% |
| 2021-08-31 | 21.56% | -8.77% | 30.09% |
| 2021-09-30 | 22.46% | -7.62% | 29.23% |
| 2021-10-29 | 19.54% | -6.81% | 25.02% |
| 2021-11-30 | 34.08% | -8.27% | 42.39% |
| 2021-12-31 | 42.51% | -6.21% | 47.84% |
| 2022-01-28 | 37.16% | -13.36% | 53.69% |
| 2022-02-28 | 47.69% | -13.02% | 64.76% |
| 2022-03-31 | 50.01% | -19.84% | 80.93% |